# 02 — Feature Engineering
## Loan Default Prediction System

**Purpose:** Apply data cleaning, derive new risk-relevant features, and validate the encoding/scaling strategy that will be assembled into the production `ColumnTransformer` used inside `loan_pipeline.pkl`.

**Input:** `ml/data/raw/Loan_default.csv`
**Output:** `ml/data/processed/loan_cleaned.csv`, `ml/data/processed/loan_engineered.csv`

This notebook calls the same `ml/src/data_cleaning.py` and `ml/src/feature_engineering.py` modules used by `train_pipeline.py`, so what is validated here is exactly what runs in production — no duplicated logic.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

ML_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = ML_ROOT / "src"
sys.path.append(str(SRC_PATH))

from data_cleaning import clean_pipeline, TARGET_COLUMN
from feature_engineering import (
    engineer_features,
    get_model_matrix,
    get_feature_columns,
    build_preprocessor,
    ENGINEERED_NUMERIC_COLUMNS,
    ENGINEERED_CATEGORICAL_COLUMNS,
)

RAW_DATA_PATH = ML_ROOT / "data" / "raw" / "Loan_default.csv"
CLEANED_DATA_PATH = ML_ROOT / "data" / "processed" / "loan_cleaned.csv"
ENGINEERED_DATA_PATH = ML_ROOT / "data" / "processed" / "loan_engineered.csv"

print(f"Raw data path: {RAW_DATA_PATH}")

## 1. Data Cleaning

Runs the full cleaning routine: missing-value imputation, duplicate removal, categorical label standardization, and IQR-based outlier capping with flag columns.

In [ ]:
cleaned_df = clean_pipeline(RAW_DATA_PATH, CLEANED_DATA_PATH)
print(f"Cleaned shape: {cleaned_df.shape}")
cleaned_df.head()

In [ ]:
outlier_flag_cols = [c for c in cleaned_df.columns if c.endswith("_was_outlier")]
print("Outlier flag columns added during cleaning:")
for col in outlier_flag_cols:
    print(f"  {col}: {int(cleaned_df[col].sum())} rows flagged")

## 2. Feature Engineering

Derives three new features on top of the cleaned data:
- **LoanToIncomeRatio** = LoanAmount / Income
- **EmploymentStabilityRatio** = MonthsEmployed / working-eligible months since age 18, bounded to [0, 1]
- **CreditScoreBand** = FICO-aligned bucket (Poor / Fair / Good / Very Good / Excellent)

In [ ]:
engineered_df = engineer_features(cleaned_df)
engineered_df.to_csv(ENGINEERED_DATA_PATH, index=False)
print(f"Engineered shape: {engineered_df.shape}")
engineered_df[["Income", "LoanAmount", "LoanToIncomeRatio",
               "Age", "MonthsEmployed", "EmploymentStabilityRatio",
               "CreditScore", "CreditScoreBand"]].head(10)

## 3. Validate New Features vs. Target

Confirming the engineered features carry meaningful separating signal before they're trusted in modeling.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(x="Default", y="LoanToIncomeRatio", data=engineered_df, ax=axes[0], palette=["#2563eb", "#dc2626"])
axes[0].set_xticklabels(["No Default", "Default"])
axes[0].set_title("Loan-to-Income Ratio by Default Status")

sns.boxplot(x="Default", y="EmploymentStabilityRatio", data=engineered_df, ax=axes[1], palette=["#2563eb", "#dc2626"])
axes[1].set_xticklabels(["No Default", "Default"])
axes[1].set_title("Employment Stability Ratio by Default Status")
plt.tight_layout()
plt.show()

In [ ]:
band_default_rate = engineered_df.groupby("CreditScoreBand")["Default"].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=band_default_rate.index, y=band_default_rate.values, color="#dc2626", ax=ax)
ax.set_ylabel("Default Rate")
ax.set_title("Default Rate by Credit Score Band")
plt.tight_layout()
plt.show()
band_default_rate

## 4. Feature Column Summary

The exact numeric/categorical column lists that will be fed into the `ColumnTransformer` at training time.

In [ ]:
columns = get_feature_columns(engineered_df)
print("Numeric columns ({}):".format(len(columns["numeric"])))
for c in columns["numeric"]:
    print(f"  - {c}")
print()
print("Categorical columns ({}):".format(len(columns["categorical"])))
for c in columns["categorical"]:
    print(f"  - {c}")

## 5. Preprocessor Dry Run

Build and fit the `ColumnTransformer` (imputers + scaler + one-hot encoder) on the full feature matrix as a sanity check — confirming no errors and inspecting the resulting transformed feature space size.

In [ ]:
X = get_model_matrix(engineered_df)
y = engineered_df[TARGET_COLUMN]

preprocessor = build_preprocessor(X)
X_transformed = preprocessor.fit_transform(X)

print(f"Original feature matrix shape: {X.shape}")
print(f"Transformed feature matrix shape: {X_transformed.shape}")
print(f"Output feature names (first 15): {list(preprocessor.get_feature_names_out()[:15])}")

## 6. Summary

- Cleaning and feature engineering ran end-to-end via the shared `ml/src/` modules with no errors.
- `LoanToIncomeRatio` and `EmploymentStabilityRatio` both show visible separation between Default and No-Default groups, confirming they add predictive signal beyond the raw features.
- `CreditScoreBand` shows a clear monotonic default-rate trend (Poor > Fair > Good > Very Good > Excellent), validating the bucket boundaries.
- The `ColumnTransformer` fits without error and expands the feature space via one-hot encoding as expected — this exact transformer becomes stage one of the final `Pipeline` saved to `loan_pipeline.pkl`.

**Next step:** `03_model_training.ipynb` — stratified train/test split, candidate model comparison, and hyperparameter tuning.